<a href="https://colab.research.google.com/github/harini200614/Data-Visualization-lab/blob/main/DVT_Lab3_231401033.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Import Libraries and Load the Real Kaggle Dataset

In [1]:

# Install KaggleHub in Google Colab
!pip -q install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os

# Download the real Kaggle dataset
dataset_path = kagglehub.dataset_download(
    "adrianjuliusaluoch/daily-google-search-trends-us"
)

# Find CSV automatically
csv_files = glob.glob(
    os.path.join(dataset_path, "**", "*.csv"),
    recursive=True
)

print("Dataset downloaded to:")
print(dataset_path)
print("\nCSV file found:")
print(csv_files[0])

df = pd.read_csv(csv_files[0])

print("\nOriginal dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())


100%|██████████| 935k/935k [00:00<00:00, 90.7MB/s]

Extracting files...
Dataset downloaded to:
/root/.cache/kagglehub/datasets/adrianjuliusaluoch/daily-google-search-trends-us/versions/194

CSV file found:
/root/.cache/kagglehub/datasets/adrianjuliusaluoch/daily-google-search-trends-us/versions/194/trending_searches_in_us.csv

Original dataset shape: (31804, 8)

Column names:
['query', 'start_date', 'end_date', 'active', 'search_volume', 'increase_percentage', 'categories', 'trend_breakdown']

First 5 records:


,query,start_date,end_date,active,search_volume,increase_percentage,categories,trend_breakdown
0,shedeur sanders,2026-03-30 18:40:00+00:00,NaN,True,200,50,Sports,NaN
1,whoopi goldberg,2026-03-30 18:40:00+00:00,NaN,True,5000,50,Politics,"america first award, mike johnson"
2,mo williams,2026-03-30 18:00:00+00:00,NaN,True,5000,300,Sports,"mason williams basketball, mason williams"
3,sloane stephens,2026-03-30 17:50:00+00:00,NaN,True,1000,200,"Sports, Health",NaN
4,canceled tv shows 2026,2026-03-30 17:50:00+00:00,NaN,True,5000,200,Entertainment,renewed and cancelled tv shows 2026


## 1. Identifying Missing Values

In [ ]:

# Boolean mask for missing entries
missing_mask = df.isnull()

print("Missing-value mask (first 5 rows):")
display(missing_mask.head())


## 2. Count Missing Values

In [ ]:

missing_counts = df.isnull().sum()

print("Missing values per column:")
print(missing_counts)

plt.figure(figsize=(12, 5))
sns.barplot(
    x=missing_counts.index,
    y=missing_counts.values
)
plt.title("Missing Values Count per Column")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 3. Replace Missing Values

In [ ]:

from sklearn.impute import SimpleImputer

# Identify numeric and categorical columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

# Numerical columns -> mean
if num_cols:
    num_imputer = SimpleImputer(strategy="mean")
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Categorical columns -> most frequent value
if cat_cols:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("\nMissing values after replacement:")
print(df.isnull().sum())


## 4. Remove Duplicate Records

In [ ]:

duplicate_count = df.duplicated().sum()

print("Total duplicate records found:", duplicate_count)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Dataset shape after removing duplicates:", df.shape)


## 5. Detecting and Capping Outliers using IQR

In [ ]:

numeric_features = df.select_dtypes(include=np.number).columns.tolist()

print("Numeric features:", numeric_features)

# Save a copy so we can compare before and after capping
df_before_outlier_capping = df.copy()

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    print(f"\nColumn: {col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Outliers detected:", outliers)

    # IQR capping
    df[col] = np.where(
        df[col] < lower_bound,
        lower_bound,
        df[col]
    )

    df[col] = np.where(
        df[col] > upper_bound,
        upper_bound,
        df[col]
    )

print("\nOutlier detection and IQR capping completed.")


### Outlier Visualization

In [ ]:

if numeric_features:
    # Plot the first numeric feature
    plot_col = numeric_features[0]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.boxplot(y=df_before_outlier_capping[plot_col])
    plt.title("Before Outlier Capping")
    plt.ylabel(plot_col)

    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[plot_col])
    plt.title("After IQR Outlier Capping")
    plt.ylabel(plot_col)

    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns available for box plot.")


## 6. Correcting Incorrect Data Types

In [ ]:

print("Initial Data Types:")
print(df.dtypes)

# Convert columns containing date/time information
for col in df.columns:
    if "date" in col.lower() or "time" in col.lower():
        converted = pd.to_datetime(df[col], errors="coerce")

        # Only replace if conversion produced useful datetime values
        if converted.notna().sum() > 0:
            df[col] = converted

print("\nUpdated Data Types:")
print(df.dtypes)


## 7. Resolving Inconsistent Formatting

In [ ]:

string_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Text columns being normalized:", string_cols)

for col in string_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

print("\nFormatted text columns sample:")
if string_cols:
    display(df[string_cols].head())
else:
    print("No text columns found.")


## 8. Exploratory Visualization

In [ ]:

# Category/text frequency visualization
if string_cols:
    plot_col = string_cols[0]

    top_values = df[plot_col].value_counts().head(10)

    plt.figure(figsize=(10, 5))
    sns.barplot(
        x=top_values.index,
        y=top_values.values
    )
    plt.title(f"Top 10 Values – {plot_col}")
    plt.xlabel(plot_col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No categorical/text column available for count plot.")


## Final Cleaned Dataset

In [ ]:

print("Final cleaned dataset shape:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)

print("\nFirst 5 rows of cleaned dataset:")
display(df.head())
